# Inspect Precomputed CDR Metrics

This notebook checks parquet files generated by `preprocess/precompute_decoy_metrics.py` under `cdr-data/metrics_precomputed`.

It focuses on quick validation:
- which source files exist
- row counts and columns
- identity key uniqueness: `target_id + source + seed + sample`
- target-level CDR counts
- metric ranges, NaN rates, and success rates
- suspicious rows such as high missing backbone atom counts

In [ ]:
from pathlib import Path
import math
import os

os.environ.setdefault("MPLCONFIGDIR", "/tmp/cdr_scoring_matplotlib")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

METRICS_ROOT = Path("/home/sujin/projects/cdr-scoring/cdr-data/metrics_precomputed")
METRICS_ROOT

In [ ]:
parquet_files = sorted(METRICS_ROOT.glob("*/*/*.parquet"))
files_df = pd.DataFrame({
    "source": [p.parts[-3] for p in parquet_files],
    "group": [p.parts[-2] for p in parquet_files],
    "file": [p.name for p in parquet_files],
    "path": [str(p) for p in parquet_files],
    "size_mb": [p.stat().st_size / 1024**2 for p in parquet_files],
})
display(files_df.sort_values(["source", "group", "file"]))

if files_df.empty:
    raise FileNotFoundError(f"No parquet files found under {METRICS_ROOT}")

In [ ]:
def read_source_table(source: str, group: str, filename: str) -> pd.DataFrame:
    path = METRICS_ROOT / source / group / filename
    if not path.exists():
        return pd.DataFrame()
    return pd.read_parquet(path)

sources = sorted(files_df["source"].unique())
tables = {}
for source in sources:
    tables[(source, "targets")] = read_source_table(source, "targets", "target_metrics.parquet")
    tables[(source, "loop")] = read_source_table(source, "metrics", "loop_metrics.parquet")
    tables[(source, "interface")] = read_source_table(source, "metrics", "interface_metrics.parquet")
    tables[(source, "dockq")] = read_source_table(source, "metrics", "dockq_metrics.parquet")

overview_rows = []
for (source, table_name), df in tables.items():
    if df.empty:
        continue
    overview_rows.append({
        "source": source,
        "table": table_name,
        "rows": len(df),
        "columns": len(df.columns),
        "targets": df["target_id"].nunique() if "target_id" in df else np.nan,
        "decoys": len(df[["target_id", "source", "seed", "sample"]].drop_duplicates()) if {"target_id", "source", "seed", "sample"}.issubset(df.columns) else np.nan,
    })

overview = pd.DataFrame(overview_rows).sort_values(["source", "table"])
display(overview)

## Inspect One Source

Change `SOURCE` below to inspect a specific source in detail.

In [ ]:
SOURCE = sources[0]
SOURCE

In [ ]:
target_df = tables[(SOURCE, "targets")]
loop_df = tables[(SOURCE, "loop")]

print("target_metrics columns:")
display(pd.DataFrame({"column": target_df.columns, "dtype": [str(t) for t in target_df.dtypes]}))
display(target_df.head(10))

print("loop_metrics columns:")
display(pd.DataFrame({"column": loop_df.columns, "dtype": [str(t) for t in loop_df.dtypes]}))
display(loop_df.head(10))

## Key Checks

The expected decoy lookup key is `target_id + source + seed + sample`.

In [ ]:
KEY = ["target_id", "source", "seed", "sample"]

key_check_rows = []
duplicate_examples = {}
for source in sources:
    df = tables[(source, "loop")]
    if df.empty:
        continue
    missing_key_cols = [c for c in KEY if c not in df.columns]
    if missing_key_cols:
        key_check_rows.append({"source": source, "rows": len(df), "missing_key_cols": missing_key_cols})
        continue
    dup_mask = df.duplicated(KEY, keep=False)
    key_check_rows.append({
        "source": source,
        "rows": len(df),
        "unique_keys": len(df[KEY].drop_duplicates()),
        "duplicate_key_rows": int(dup_mask.sum()),
        "missing_seed_rows": int(df["seed"].isna().sum()),
        "missing_sample_rows": int(df["sample"].isna().sum()),
    })
    if dup_mask.any():
        duplicate_examples[source] = df.loc[dup_mask, KEY + ["decoy_id", "decoy_path"]].head(20)

display(pd.DataFrame(key_check_rows))
for source, dup_df in duplicate_examples.items():
    print(f"Duplicate examples: {source}")
    display(dup_df)

## Target-Level CDR Counts

In [ ]:
count_cols = ["H1_count", "H2_count", "H3_count", "L1_count", "L2_count", "L3_count"]
target_summary_rows = []
for source in sources:
    df = tables[(source, "targets")]
    if df.empty:
        continue
    row = {"source": source, "targets": len(df), "unique_targets": df["target_id"].nunique()}
    for col in count_cols:
        if col in df:
            row[f"{col}_min"] = df[col].min()
            row[f"{col}_median"] = df[col].median()
            row[f"{col}_max"] = df[col].max()
            row[f"{col}_zero"] = int((df[col] == 0).sum())
    target_summary_rows.append(row)

target_summary = pd.DataFrame(target_summary_rows)
display(target_summary)

if not target_df.empty:
    display(target_df[["target_id", "source", "has_holo_antigen", "antibody_chains", "antigen_chains"] + count_cols].head(20))

## Loop Metric Sanity Summary

In [ ]:
metric_cols = [
    "global_loop_rmsd", "global_loop_lddt",
    "H1_loop_rmsd", "H2_loop_rmsd", "H3_loop_rmsd", "L1_loop_rmsd", "L2_loop_rmsd", "L3_loop_rmsd",
    "H1_loop_lddt", "H2_loop_lddt", "H3_loop_lddt", "L1_loop_lddt", "L2_loop_lddt", "L3_loop_lddt",
]

summary_rows = []
for source in sources:
    df = tables[(source, "loop")]
    if df.empty:
        continue
    row = {"source": source, "rows": len(df), "targets": df["target_id"].nunique()}
    for col in ["global_loop_rmsd", "global_loop_lddt"]:
        values = pd.to_numeric(df[col], errors="coerce") if col in df else pd.Series(dtype=float)
        finite = values[np.isfinite(values)]
        row[f"{col}_finite"] = int(finite.size)
        row[f"{col}_nan"] = int(values.isna().sum())
        row[f"{col}_min"] = finite.min() if finite.size else np.nan
        row[f"{col}_median"] = finite.median() if finite.size else np.nan
        row[f"{col}_max"] = finite.max() if finite.size else np.nan
    if "global_loop_rmsd" in df:
        rmsd = pd.to_numeric(df["global_loop_rmsd"], errors="coerce")
        row["success_rmsd_le_2"] = float((rmsd <= 2.0).mean())
    if "global_loop_lddt" in df:
        lddt = pd.to_numeric(df["global_loop_lddt"], errors="coerce")
        row["success_lddt_ge_0p8"] = float((lddt >= 0.8).mean())
    if "missing_backbone_atom_count" in df:
        row["missing_backbone_ge_5"] = int((df["missing_backbone_atom_count"] >= 5).sum())
    summary_rows.append(row)

loop_summary = pd.DataFrame(summary_rows)
display(loop_summary)

In [ ]:
if not loop_df.empty:
    print("Rows with missing backbone atom count >= 5")
    cols = ["target_id", "source", "seed", "sample", "decoy_id", "missing_backbone_atom_count", "missing_backbone_report", "decoy_path"]
    display(loop_df.loc[loop_df["missing_backbone_atom_count"] >= 5, cols].head(50))

    print("Rows with NaN global loop RMSD or lDDT")
    nan_mask = loop_df["global_loop_rmsd"].isna() | loop_df["global_loop_lddt"].isna()
    display(loop_df.loc[nan_mask, ["target_id", "source", "seed", "sample", "decoy_id", "global_loop_rmsd", "global_loop_lddt", "missing_backbone_report", "decoy_path"]].head(50))

## Metric Distributions

In [ ]:
all_loop = []
for source in sources:
    df = tables[(source, "loop")]
    if not df.empty:
        all_loop.append(df.assign(source=source))
all_loop = pd.concat(all_loop, ignore_index=True) if all_loop else pd.DataFrame()

if not all_loop.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for source, sub in all_loop.groupby("source"):
        axes[0].hist(sub["global_loop_rmsd"].dropna(), bins=50, alpha=0.45, label=source)
        axes[1].hist(sub["global_loop_lddt"].dropna(), bins=50, alpha=0.45, label=source)
    axes[0].set_title("global_loop_rmsd")
    axes[0].set_xlabel("A")
    axes[0].set_ylabel("count")
    axes[1].set_title("global_loop_lddt")
    axes[1].set_xlabel("lDDT")
    axes[1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("No loop metrics loaded")

In [ ]:
if not all_loop.empty:
    per_cdr_rmsd = ["H1_loop_rmsd", "H2_loop_rmsd", "H3_loop_rmsd", "L1_loop_rmsd", "L2_loop_rmsd", "L3_loop_rmsd"]
    per_cdr_lddt = ["H1_loop_lddt", "H2_loop_lddt", "H3_loop_lddt", "L1_loop_lddt", "L2_loop_lddt", "L3_loop_lddt"]
    display(all_loop.groupby("source")[per_cdr_rmsd + per_cdr_lddt].agg(["count", "median", "min", "max"]))

## Inspect One Target

In [ ]:
if not loop_df.empty:
    TARGET_ID = loop_df["target_id"].iloc[0]
    sub = loop_df[loop_df["target_id"] == TARGET_ID].copy()
    print("SOURCE =", SOURCE, "TARGET_ID =", TARGET_ID, "n =", len(sub))
    display(target_df[target_df["target_id"] == TARGET_ID])
    display(sub.sort_values("global_loop_rmsd").head(20))

## Boltz2 Apo vs Holo Analysis

The following cells focus on Boltz2 and split decoys by the target-level `has_holo_antigen` flag from `targets/target_metrics.parquet`.

In [ ]:
boltz_source = next((s for s in sources if s.lower() == "boltz2"), None)
if boltz_source is None:
    raise ValueError("Boltz2 source was not found under METRICS_ROOT")

boltz_loop = tables[(boltz_source, "loop")].copy()
boltz_targets = tables[(boltz_source, "targets")].copy()

target_state_cols = ["target_id", "source", "has_holo_antigen", "antigen_chains"]
boltz = boltz_loop.merge(
    boltz_targets[target_state_cols].drop_duplicates(),
    on=["target_id", "source"],
    how="left",
)
boltz["target_state"] = np.where(boltz["has_holo_antigen"].fillna(False), "holo", "apo")

print("Boltz2 rows:", len(boltz), "targets:", boltz["target_id"].nunique())
display(boltz.groupby("target_state").agg(rows=("target_id", "size"), targets=("target_id", "nunique")))
display(boltz.head())

### Figure 1. Boltz2 global_loop_rmsd / global_loop_lddt distribution, apo vs holo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = {"apo": "tab:blue", "holo": "tab:orange"}

for state, sub in boltz.groupby("target_state"):
    axes[0].hist(sub["global_loop_rmsd"].dropna(), bins=40, alpha=0.55, label=f"{state} (n={len(sub)})", color=colors.get(state))
    axes[1].hist(sub["global_loop_lddt"].dropna(), bins=40, alpha=0.55, label=f"{state} (n={len(sub)})", color=colors.get(state))

axes[0].set_title("Boltz2 global_loop_rmsd")
axes[0].set_xlabel("RMSD (A)")
axes[0].set_ylabel("Decoy count")
axes[0].legend()

axes[1].set_title("Boltz2 global_loop_lddt")
axes[1].set_xlabel("lDDT")
axes[1].set_ylabel("Decoy count")
axes[1].legend()

plt.tight_layout()
plt.show()

display(boltz.groupby("target_state")[["global_loop_rmsd", "global_loop_lddt"]].agg(["count", "median", "mean", "min", "max"]))

### Figure 2. Per-CDR loop_rmsd boxplot: H1-H3/L1-L3, apo vs holo

In [ ]:
cdr_order = ["H1", "H2", "H3", "L1", "L2", "L3"]
rmsd_cols = [f"{cdr}_loop_rmsd" for cdr in cdr_order]
lddt_cols = [f"{cdr}_loop_lddt" for cdr in cdr_order]


def _plot_per_cdr_state_boxplot(metric_cols, ylabel, title):
    box_data = []
    positions = []
    box_colors = []
    pos = 1
    for col in metric_cols:
        for offset, state in enumerate(["apo", "holo"]):
            values = pd.to_numeric(boltz.loc[boltz["target_state"] == state, col], errors="coerce").dropna().to_numpy()
            box_data.append(values if len(values) else np.array([np.nan]))
            positions.append(pos + offset * 0.35)
            box_colors.append(colors.get(state, "gray"))
        pos += 1.1

    fig, ax = plt.subplots(figsize=(11, 4.5))
    bp = ax.boxplot(box_data, positions=positions, widths=0.28, patch_artist=True, showfliers=False)
    for patch, color in zip(bp["boxes"], box_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.55)

    centers = [1 + i * 1.1 + 0.175 for i in range(len(cdr_order))]
    ax.set_xticks(centers)
    ax.set_xticklabels(cdr_order)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)

    handles = [plt.Rectangle((0, 0), 1, 1, color=colors[state], alpha=0.55) for state in ["apo", "holo"]]
    ax.legend(handles, ["apo", "holo"], title="target state")
    plt.tight_layout()
    plt.show()


_plot_per_cdr_state_boxplot(rmsd_cols, "loop_rmsd (A)", "Boltz2 per-CDR loop_rmsd: apo vs holo")
_plot_per_cdr_state_boxplot(lddt_cols, "loop_lddt", "Boltz2 per-CDR loop_lddt: apo vs holo")

print("Per-CDR RMSD summary")
display(boltz.groupby("target_state")[rmsd_cols].agg(["count", "median", "mean"]))
print("Per-CDR lDDT summary")
display(boltz.groupby("target_state")[lddt_cols].agg(["count", "median", "mean"]))


### ComMat vs Boltz2 2D Density: global_loop_lddt vs H3_loop_lddt

This compares the decoy-level quality distribution of ComMat and Boltz2. Higher values are better for both axes.


In [ ]:
compare_source_names = ["ComMat", "Boltz2"]
compare_sources = []
for name in compare_source_names:
    source = next((s for s in sources if s.lower() == name.lower()), None)
    if source is None:
        print(f"Missing source: {name}")
    else:
        compare_sources.append(source)

required_cols = ["target_id", "source", "global_loop_lddt", "H3_loop_lddt"]
density_frames = []
for source in compare_sources:
    df = tables[(source, "loop")]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        print(f"Skip {source}: missing {missing}")
        continue
    sub = df[required_cols].copy()
    sub["global_loop_lddt"] = pd.to_numeric(sub["global_loop_lddt"], errors="coerce")
    sub["H3_loop_lddt"] = pd.to_numeric(sub["H3_loop_lddt"], errors="coerce")
    sub = sub.dropna(subset=["global_loop_lddt", "H3_loop_lddt"])
    density_frames.append(sub)

if not density_frames:
    raise ValueError("No ComMat/Boltz2 loop lDDT data available for density plot")

density_df = pd.concat(density_frames, ignore_index=True)
display(density_df.groupby("source").agg(rows=("target_id", "size"), targets=("target_id", "nunique")))

source_colors = {"ComMat": "tab:green", "Boltz2": "tab:purple"}
fig, ax = plt.subplots(figsize=(6.5, 5.8))

for source in compare_sources:
    sub = density_df[density_df["source"] == source]
    if sub.empty:
        continue
    x = sub["global_loop_lddt"].to_numpy(dtype=float)
    y = sub["H3_loop_lddt"].to_numpy(dtype=float)
    sample = sub.sample(min(len(sub), 2500), random_state=0)
    color = source_colors.get(source, None)
    ax.scatter(
        sample["global_loop_lddt"],
        sample["H3_loop_lddt"],
        s=8,
        alpha=0.05,
        color=color,
        rasterized=True,
    )

    hist, x_edges, y_edges = np.histogram2d(x, y, bins=60, range=[[0, 1], [0, 1]])
    positive = hist[hist > 0]
    if positive.size:
        levels = np.unique(np.quantile(positive, [0.55, 0.75, 0.90, 0.97]))
        levels = levels[levels > 0]
        if levels.size:
            x_centers = (x_edges[:-1] + x_edges[1:]) / 2
            y_centers = (y_edges[:-1] + y_edges[1:]) / 2
            ax.contour(
                x_centers,
                y_centers,
                hist.T,
                levels=levels,
                colors=[color],
                linewidths=1.7,
                alpha=0.95,
            )

legend_handles = [plt.Line2D([0], [0], color=source_colors.get(source, "black"), lw=2, label=source) for source in compare_sources]
ax.plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1, alpha=0.7)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("global_loop_lddt")
ax.set_ylabel("H3_loop_lddt")
ax.set_title("ComMat vs Boltz2 decoy quality density")
ax.grid(alpha=0.2)
ax.legend(handles=legend_handles, title="source")
plt.tight_layout()
plt.show()


### Boltz2 Top-1 Selection Quality

This target-level analysis evaluates whether the Boltz2 rank-1 decoy is actually good according to true Full-CDR `global_loop_lddt` within each target. `global_loop_lddt` is a loop-level lDDT metric; higher is better.

For each target:
1. Select Boltz2 rank-1 decoy using `ranking` or `boltz2_rank`.
2. Rank all Boltz2 decoys for that target by `global_loop_lddt`, higher is better.
3. Compute true rank, true percentile, oracle best lDDT, top1 lDDT, and oracle gap.

Saved outputs:
- `figures/boltz2_top1_hit_rate.png`
- `figures/boltz2_top1_percentile_bins.png`
- `figures/boltz2_top1_oracle_gap.png`
- `tables/boltz2_top1_target_level.csv`
- `tables/boltz2_top1_summary.csv`


In [ ]:
ANALYSIS_OUT_DIR = METRICS_ROOT / "analysis_outputs"
FIGURES_DIR = ANALYSIS_OUT_DIR / "figures"
TABLES_DIR = ANALYSIS_OUT_DIR / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
print("Analysis outputs:", ANALYSIS_OUT_DIR)


In [ ]:
def compute_boltz2_top1_target_level(decoy_df: pd.DataFrame) -> pd.DataFrame:
    required = {"target_id", "source", "global_loop_lddt"}
    missing = required - set(decoy_df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    rank_col = "boltz2_rank" if "boltz2_rank" in decoy_df.columns else "ranking"
    if rank_col not in decoy_df.columns:
        raise ValueError("Expected either ranking or boltz2_rank column")

    work = decoy_df.copy()
    work[rank_col] = pd.to_numeric(work[rank_col], errors="coerce")
    work["global_loop_lddt"] = pd.to_numeric(work["global_loop_lddt"], errors="coerce")
    if "ranking_score" in work.columns:
        work["ranking_score"] = pd.to_numeric(work["ranking_score"], errors="coerce")
    else:
        work["ranking_score"] = np.nan
    if "target_state" not in work.columns:
        if "has_holo_antigen" in work.columns:
            work["target_state"] = np.where(work["has_holo_antigen"].fillna(False), "holo", "apo")
        else:
            work["target_state"] = "unknown"

    rows = []
    for target_id, sub in work.groupby("target_id", dropna=False):
        finite = sub.dropna(subset=["global_loop_lddt"]).copy()
        ranked = finite.dropna(subset=[rank_col]).copy()
        if finite.empty or ranked.empty:
            continue

        ranked = ranked.sort_values([rank_col, "ranking_score", "sample"], ascending=[True, False, True], kind="mergesort")
        top1 = ranked.iloc[0]
        n_decoys = int(len(finite))
        top1_lddt = float(top1["global_loop_lddt"])
        oracle_best_lddt = float(finite["global_loop_lddt"].max())
        top1_true_rank = int((finite["global_loop_lddt"] > top1_lddt).sum()) + 1
        top1_true_percentile = (n_decoys - top1_true_rank + 1) / n_decoys
        oracle_gap = oracle_best_lddt - top1_lddt

        rows.append({
            "target_id": target_id,
            "source": top1.get("source", "Boltz2"),
            "target_state": top1.get("target_state", "unknown"),
            "has_holo_antigen": bool(top1.get("has_holo_antigen", False)),
            "n_decoys": n_decoys,
            "boltz2_rank_col": rank_col,
            "boltz2_top1_rank": top1.get(rank_col, np.nan),
            "boltz2_top1_sample": top1.get("sample", np.nan),
            "boltz2_top1_ranking_score": top1.get("ranking_score", np.nan),
            "top1_true_rank": top1_true_rank,
            "top1_true_percentile": float(top1_true_percentile),
            "oracle_best_lddt": oracle_best_lddt,
            "boltz2_top1_lddt": top1_lddt,
            "oracle_gap": float(oracle_gap),
        })
    return pd.DataFrame(rows)

boltz_top1_target = compute_boltz2_top1_target_level(boltz)
target_level_path = TABLES_DIR / "boltz2_top1_target_level.csv"
boltz_top1_target.to_csv(target_level_path, index=False)
print("target-level rows:", len(boltz_top1_target), "saved:", target_level_path)
display(boltz_top1_target.head(20))


In [ ]:
def _summary_for_group(df: pd.DataFrame, group: str) -> dict:
    if df.empty:
        return {
            "group": group,
            "n_targets": 0,
            "fraction_true_best": np.nan,
            "fraction_true_top5_percent": np.nan,
            "fraction_true_top10_percent": np.nan,
            "fraction_true_top25_percent": np.nan,
            "fraction_true_top50_percent": np.nan,
            "median_true_percentile": np.nan,
            "mean_true_percentile": np.nan,
            "median_oracle_gap": np.nan,
            "mean_oracle_gap": np.nan,
            "fraction_gap_lt_0.01": np.nan,
            "fraction_gap_lt_0.05": np.nan,
        }
    percentile = df["top1_true_percentile"]
    gap = df["oracle_gap"]
    return {
        "group": group,
        "n_targets": int(len(df)),
        "fraction_true_best": float((df["top1_true_rank"] == 1).mean()),
        "fraction_true_top5_percent": float((percentile >= 0.95).mean()),
        "fraction_true_top10_percent": float((percentile >= 0.90).mean()),
        "fraction_true_top25_percent": float((percentile >= 0.75).mean()),
        "fraction_true_top50_percent": float((percentile >= 0.50).mean()),
        "median_true_percentile": float(percentile.median()),
        "mean_true_percentile": float(percentile.mean()),
        "median_oracle_gap": float(gap.median()),
        "mean_oracle_gap": float(gap.mean()),
        "fraction_gap_lt_0.01": float((gap < 0.01).mean()),
        "fraction_gap_lt_0.05": float((gap < 0.05).mean()),
    }

summary_rows = [_summary_for_group(boltz_top1_target, "all")]
for state in ["apo", "holo"]:
    summary_rows.append(_summary_for_group(boltz_top1_target[boltz_top1_target["target_state"] == state], state))

boltz_top1_summary = pd.DataFrame(summary_rows)
summary_path = TABLES_DIR / "boltz2_top1_summary.csv"
boltz_top1_summary.to_csv(summary_path, index=False)
print("saved:", summary_path)
display(boltz_top1_summary)


### Figure A. Top-k Hit Rate Bar Plot

Fraction of targets where Boltz2 rank-1 falls within the true-quality group defined by Full-CDR `global_loop_lddt`.


In [ ]:
hit_cols = [
    "fraction_true_best",
    "fraction_true_top5_percent",
    "fraction_true_top10_percent",
    "fraction_true_top25_percent",
    "fraction_true_top50_percent",
]
hit_labels = ["true best", "true top 5%", "true top 10%", "true top 25%", "true top 50%"]
groups = ["all", "apo", "holo"]
group_colors = {"all": "black", "apo": colors.get("apo", "tab:blue"), "holo": colors.get("holo", "tab:orange")}

x = np.arange(len(hit_cols))
width = 0.25
fig, ax = plt.subplots(figsize=(10, 5))
for i, group in enumerate(groups):
    row = boltz_top1_summary[boltz_top1_summary["group"] == group].iloc[0]
    values = [row[col] for col in hit_cols]
    ax.bar(x + (i - 1) * width, values, width=width, label=f"{group} (n={int(row['n_targets'])})", color=group_colors[group], alpha=0.75)

ax.set_xticks(x)
ax.set_xticklabels(hit_labels, rotation=20, ha="right")
ax.set_ylim(0, 1.0)
ax.set_ylabel("Fraction of targets")
ax.set_title("Figure A. Boltz2 top-1 true-quality hit rate")
ax.grid(axis="y", alpha=0.25)
ax.legend(title="target group")
plt.tight_layout()
fig_path = FIGURES_DIR / "boltz2_top1_hit_rate.png"
plt.savefig(fig_path, dpi=200)
plt.show()
print("saved:", fig_path)


### Figure A2. Top-1 Hit Rate by Loop Metric

This repeats the same top-1 selection-quality analysis for global and per-CDR loop lDDT/RMSD. lDDT uses higher-is-better ranking; RMSD uses lower-is-better ranking.


In [ ]:
def compute_boltz2_top1_target_level_for_metric(decoy_df: pd.DataFrame, metric_col: str, higher_is_better: bool) -> pd.DataFrame:
    required = {"target_id", "source", metric_col}
    missing = required - set(decoy_df.columns)
    if missing:
        raise ValueError(f"Missing required columns for {metric_col}: {sorted(missing)}")

    rank_col = "boltz2_rank" if "boltz2_rank" in decoy_df.columns else "ranking"
    if rank_col not in decoy_df.columns:
        raise ValueError("Expected either ranking or boltz2_rank column")

    work = decoy_df.copy()
    work[rank_col] = pd.to_numeric(work[rank_col], errors="coerce")
    work[metric_col] = pd.to_numeric(work[metric_col], errors="coerce")
    if "ranking_score" in work.columns:
        work["ranking_score"] = pd.to_numeric(work["ranking_score"], errors="coerce")
    else:
        work["ranking_score"] = np.nan
    if "target_state" not in work.columns:
        if "has_holo_antigen" in work.columns:
            work["target_state"] = np.where(work["has_holo_antigen"].fillna(False), "holo", "apo")
        else:
            work["target_state"] = "unknown"

    sort_cols = [rank_col]
    ascending = [True]
    if "ranking_score" in work.columns:
        sort_cols.append("ranking_score")
        ascending.append(False)
    if "sample" in work.columns:
        sort_cols.append("sample")
        ascending.append(True)

    rows = []
    for target_id, sub in work.groupby("target_id", dropna=False):
        finite = sub.dropna(subset=[metric_col]).copy()
        ranked = finite.dropna(subset=[rank_col]).copy()
        if finite.empty or ranked.empty:
            continue

        ranked = ranked.sort_values(sort_cols, ascending=ascending, kind="mergesort")
        top1 = ranked.iloc[0]
        values = finite[metric_col]
        n_decoys = int(len(finite))
        top1_value = float(top1[metric_col])
        if higher_is_better:
            oracle_best_value = float(values.max())
            top1_true_rank = int((values > top1_value).sum()) + 1
            oracle_gap = oracle_best_value - top1_value
        else:
            oracle_best_value = float(values.min())
            top1_true_rank = int((values < top1_value).sum()) + 1
            oracle_gap = top1_value - oracle_best_value
        top1_true_percentile = (n_decoys - top1_true_rank + 1) / n_decoys

        rows.append({
            "target_id": target_id,
            "source": top1.get("source", "Boltz2"),
            "target_state": top1.get("target_state", "unknown"),
            "metric": metric_col,
            "higher_is_better": bool(higher_is_better),
            "n_decoys": n_decoys,
            "boltz2_top1_sample": top1.get("sample", np.nan),
            "boltz2_top1_ranking_score": top1.get("ranking_score", np.nan),
            "top1_true_rank": top1_true_rank,
            "top1_true_percentile": float(top1_true_percentile),
            "oracle_best_value": oracle_best_value,
            "boltz2_top1_value": top1_value,
            "oracle_gap": float(oracle_gap),
        })
    return pd.DataFrame(rows)

metric_specs = []
for prefix, label_prefix, higher in [
    ("loop_lddt", "lDDT", True),
    ("loop_rmsd", "RMSD", False),
]:
    metric_specs.append({
        "metric": f"global_{prefix}",
        "label": f"global {label_prefix}",
        "higher_is_better": higher,
    })
    for cdr in cdr_order:
        metric_specs.append({
            "metric": f"{cdr}_{prefix}",
            "label": f"{cdr} {label_prefix}",
            "higher_is_better": higher,
        })

metric_target_tables = []
metric_summary_rows = []
for spec in metric_specs:
    if spec["metric"] not in boltz.columns:
        print(f"Skip missing metric: {spec['metric']}")
        continue
    metric_target = compute_boltz2_top1_target_level_for_metric(
        boltz,
        metric_col=spec["metric"],
        higher_is_better=spec["higher_is_better"],
    )
    metric_target["metric_label"] = spec["label"]
    metric_target_tables.append(metric_target)

    for group in groups:
        sub = metric_target if group == "all" else metric_target[metric_target["target_state"] == group]
        row = _summary_for_group(sub, group)
        row.update({
            "metric": spec["metric"],
            "metric_label": spec["label"],
            "higher_is_better": spec["higher_is_better"],
        })
        metric_summary_rows.append(row)

boltz_metric_top1_target = pd.concat(metric_target_tables, ignore_index=True) if metric_target_tables else pd.DataFrame()
boltz_metric_top1_summary = pd.DataFrame(metric_summary_rows)
metric_target_path = TABLES_DIR / "boltz2_top1_by_metric_target_level.csv"
metric_summary_path = TABLES_DIR / "boltz2_top1_by_metric_summary.csv"
boltz_metric_top1_target.to_csv(metric_target_path, index=False)
boltz_metric_top1_summary.to_csv(metric_summary_path, index=False)
print("saved:", metric_target_path)
print("saved:", metric_summary_path)

display(boltz_metric_top1_summary[
    ["metric_label", "group", "n_targets", "fraction_true_best", "fraction_true_top10_percent", "fraction_true_top25_percent", "median_true_percentile", "median_oracle_gap"]
].head(30))

plot_cols = [
    "fraction_true_best",
    "fraction_true_top5_percent",
    "fraction_true_top10_percent",
    "fraction_true_top25_percent",
    "fraction_true_top50_percent",
]
plot_labels = ["best", "top 5%", "top 10%", "top 25%", "top 50%"]
metric_order = [spec["label"] for spec in metric_specs if spec["metric"] in boltz.columns]

fig, axes = plt.subplots(1, 3, figsize=(15, 8), sharey=True, constrained_layout=True)
for ax, group in zip(axes, groups):
    sub = boltz_metric_top1_summary[boltz_metric_top1_summary["group"] == group].set_index("metric_label")
    matrix = sub.reindex(metric_order)[plot_cols].to_numpy(dtype=float)
    im = ax.imshow(matrix, vmin=0, vmax=1, cmap="viridis", aspect="auto")
    ax.set_title(group)
    ax.set_xticks(np.arange(len(plot_cols)))
    ax.set_xticklabels(plot_labels, rotation=35, ha="right")
    ax.set_yticks(np.arange(len(metric_order)))
    ax.set_yticklabels(metric_order if ax is axes[0] else [])
    for y in range(matrix.shape[0]):
        for x_idx in range(matrix.shape[1]):
            value = matrix[y, x_idx]
            if np.isfinite(value):
                ax.text(x_idx, y, f"{value:.2f}", ha="center", va="center", fontsize=7, color="white" if value < 0.55 else "black")

fig.suptitle("Figure A2. Boltz2 rank-1 true-quality hit rate by evaluation metric", y=1.01)
fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02, label="Fraction of targets")
fig_path = FIGURES_DIR / "boltz2_top1_metric_hit_rate.png"
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()
print("saved:", fig_path)


### Figure B. Three-bin Percentile Summary Bar Plot

Bins: poor `0-50%`, okay `50-90%`, good `90-100%`.

In [ ]:
def _percentile_bins(df: pd.DataFrame) -> dict:
    p = df["top1_true_percentile"]
    n = len(p)
    if n == 0:
        return {"poor": np.nan, "okay": np.nan, "good": np.nan}
    return {
        "poor": float((p < 0.50).mean()),
        "okay": float(((p >= 0.50) & (p < 0.90)).mean()),
        "good": float((p >= 0.90).mean()),
    }

bin_rows = []
for group in groups:
    sub = boltz_top1_target if group == "all" else boltz_top1_target[boltz_top1_target["target_state"] == group]
    row = {"group": group, "n_targets": len(sub)}
    row.update(_percentile_bins(sub))
    bin_rows.append(row)
bin_df = pd.DataFrame(bin_rows)
display(bin_df)

fig, ax = plt.subplots(figsize=(7, 5))
bottom = np.zeros(len(bin_df))
bin_colors = {"poor": "#bdbdbd", "okay": "#6baed6", "good": "#31a354"}
bin_labels = {
    "poor": "poor: 0-50%",
    "okay": "okay: 50-90%",
    "good": "good: 90-100%",
}
for bin_name in ["poor", "okay", "good"]:
    values = bin_df[bin_name].to_numpy(dtype=float)
    ax.bar(bin_df["group"], values, bottom=bottom, label=bin_labels[bin_name], color=bin_colors[bin_name], alpha=0.85)
    bottom += np.nan_to_num(values)

ax.set_ylim(0, 1.0)
ax.set_ylabel("Fraction of targets")
ax.set_title("Figure B. Boltz2 top-1 true-percentile bins")
ax.legend(title="top1 true percentile")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
fig_path = FIGURES_DIR / "boltz2_top1_percentile_bins.png"
plt.savefig(fig_path, dpi=200)
plt.show()
print("saved:", fig_path)


### Figure C. Oracle Gap Plot

`oracle_gap = oracle_best_lddt - boltz2_top1_lddt`. Lower is better; zero means Boltz2 rank-1 is oracle-best by Full-CDR global loop lDDT.

In [ ]:
gap_states = ["apo", "holo"]
gap_data = [
    boltz_top1_target.loc[boltz_top1_target["target_state"] == state, "oracle_gap"].dropna().to_numpy()
    for state in gap_states
]

fig, ax = plt.subplots(figsize=(6, 5))
bp = ax.boxplot(gap_data, labels=gap_states, patch_artist=True, showfliers=False)
for patch, state in zip(bp["boxes"], gap_states):
    patch.set_facecolor(colors.get(state, "gray"))
    patch.set_alpha(0.65)

rng = np.random.default_rng(0)
for x_pos, values in enumerate(gap_data, start=1):
    if len(values) == 0:
        continue
    jitter = rng.normal(loc=0.0, scale=0.025, size=len(values))
    ax.scatter(np.full(len(values), x_pos) + jitter, values, s=10, alpha=0.22, color="black")

ax.axhline(0, color="gray", linewidth=1)
ax.set_xlabel("target state")
ax.set_ylabel("oracle_best_lddt - boltz2_top1_lddt")
ax.set_title("Figure C. Boltz2 top-1 oracle gap")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
fig_path = FIGURES_DIR / "boltz2_top1_oracle_gap.png"
plt.savefig(fig_path, dpi=200)
plt.show()
print("saved:", fig_path)

gap_summary_cols = ["group", "n_targets", "median_oracle_gap", "mean_oracle_gap", "fraction_gap_lt_0.01", "fraction_gap_lt_0.05"]
display(boltz_top1_summary[gap_summary_cols])
